## 环境安装

如果运行环境缺少依赖，先运行下面这个单元。`py3Dmol` 用于显示分子结构，`tqdm` 用于显示训练进度条。


In [ ]:
!pip install py3Dmol tqdm

# Toy MLIP Reliability Demo: 从势能面拟合到 OOD 诊断

**AI4S 公开课实战：训练一个 toy MLIP，并判断它什么时候不可信**

机器学习势函数直接学习从原子种类和坐标到能量的映射。模型输出能量后，可以通过能量对坐标的梯度得到原子力。

在真实材料模拟中，MLIP 的难点往往不只是把 ID 测试误差做低，而是判断它在新温压、新缺陷、新反应路径等训练数据覆盖之外的区域是否仍然可信。本实战用一个三维水分子 toy system，把这个可靠性诊断流程压缩到 15–20 分钟内完成。

## 学习目标

1. 训练一个轻量神经网络势函数，学习构型到能量和力的映射。
2. 检查能量旋转不变性与力旋转等变性，理解物理对称性为什么是 sanity check。
3. 比较 ID / OOD 误差，并用 committee disagreement 与数据回流模拟主动学习闭环。

本 notebook 不使用真实 DFT 数据，而是用解析 toy potential 生成“伪 DFT”能量和力标签。目标不是追求最高精度，而是理解 MLIP 可靠性诊断的基本工作流。



## Section 1. 数据模块：生成三维水分子 toy 势能面

这一节先构造一个可控的“伪 DFT”世界。每个构型是一个三维水分子，能量由两个 O-H 键长、H-O-H 角度和 H-H 排斥项决定，参考力由能量自动微分得到。

我们会生成三类数据视角：训练用的 ID 构型、接近平衡的 ID 测试构型，以及四类 OOD 构型。后面的所有可靠性分析，都建立在“训练数据覆盖了哪里、没有覆盖哪里”这个问题上。

这里的 OOD 指 out-of-distribution，也就是训练分布之外的构型。本 demo 里四类 OOD 分别对应：

- **OOD_highT**：键长和键角都有更大扰动，模拟高温下更剧烈的热振动。
- **OOD_stretch**：一个 O-H 键被明显拉长，模拟拉伸、断键前后的构型。
- **OOD_compress**：一个 O-H 键被明显压缩，模拟原子距离过近的高排斥区域。
- **OOD_angle**：H-O-H 角度明显偏离平衡值，模拟模型没有充分见过的弯曲构型。



先导入依赖、设置随机种子和绘图参数。这样每次运行都会从同一组随机数开始，便于复现实验结果。




In [ ]:
import os
import math
import random
import json
from pathlib import Path
import base64
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import py3Dmol
from IPython.display import display
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
DTYPE = torch.float32
plt.rcParams.update({
    "figure.dpi": 120,
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 14,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 10,
})

# Keep model colors consistent across all plots: red for RawCoordNet, blue for InvariantFeatureNet.
MODEL_COLORS = {
    "RawCoordNet": "#df2a18",
    "InvariantFeatureNet": "#61b7df",
}

BAR_COLORS = {
    "RawCoordNet": MODEL_COLORS["RawCoordNet"],
    "InvariantFeatureNet": MODEL_COLORS["InvariantFeatureNet"],
    "before": "#df2a18",
    "after": "#61b7df",
}

OUTPUT_DIR = Path("outputs")
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig, filename):
    """Save a classroom preview figure under outputs/figures."""
    path = FIGURE_DIR / filename
    fig.savefig(path, dpi=180, bbox_inches="tight")
    print(f"Saved figure: {path}")
    return path


def label_log_bars(ax, bars, fmt="{:.2g}", factor=1.12, fontsize=8):
    """Add value labels above bars on a log-scaled y axis."""
    values = []
    for bar in bars:
        value = float(bar.get_height())
        values.append(value)
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            max(value, 1e-12) * factor,
            fmt.format(value),
            ha="center",
            va="bottom",
            fontsize=fontsize,
            color="black",
            fontfamily="DejaVu Serif",
        )
    return values


def expand_log_ylim_for_labels(ax, values, factor=2.2):
    """Leave enough headroom for bar labels on a log-scaled y axis."""
    positive = [float(v) for v in values if float(v) > 0]
    if positive:
        bottom, _ = ax.get_ylim()
        ax.set_ylim(bottom=bottom, top=max(positive) * factor)


def apply_reference_bar_style(ax, title, ylabel, x, labels, rotation=0):
    """Apply the requested clean serif bar-chart style."""
    ax.set_title(title, fontfamily="DejaVu Serif", fontsize=18, pad=8)
    ax.set_ylabel(ylabel, fontfamily="DejaVu Serif", fontsize=16)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=rotation, ha="right" if rotation else "center")
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontfamily("DejaVu Serif")
        tick.set_fontsize(11 if tick in ax.get_xticklabels() else 12)
        tick.set_fontweight("normal")
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(1.8)
        ax.spines[side].set_color("black")
    ax.tick_params(axis="both", which="major", width=1.6, length=5, color="black")
    ax.tick_params(axis="both", which="minor", width=1.1, length=3, color="black")


def apply_reference_axis_style(ax, title, xlabel, ylabel):
    """Apply the same clean serif axis style to non-bar 2D plots."""
    ax.set_title(title, fontfamily="DejaVu Serif", fontsize=18, pad=8)
    ax.set_xlabel(xlabel, fontfamily="DejaVu Serif", fontsize=16)
    ax.set_ylabel(ylabel, fontfamily="DejaVu Serif", fontsize=16)
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontfamily("DejaVu Serif")
        tick.set_fontsize(12)
        tick.set_fontweight("normal")
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_linewidth(1.8)
        ax.spines[side].set_color("black")
    ax.tick_params(axis="both", which="major", width=1.6, length=5, color="black")
    ax.tick_params(axis="both", which="minor", width=1.1, length=3, color="black")


def apply_reference_3d_style(fig, ax, title, xlabel, ylabel, zlabel):
    """Make 3D diagnostic plots visually consistent with the 2D chart style."""
    fig.suptitle(title, y=0.935, fontfamily="DejaVu Serif", fontsize=17)
    ax.set_xlabel(xlabel, fontfamily="DejaVu Serif", fontsize=11, labelpad=3)
    ax.set_ylabel(ylabel, fontfamily="DejaVu Serif", fontsize=11, labelpad=3)
    ax.set_zlabel(zlabel, fontfamily="DejaVu Serif", fontsize=11, labelpad=1)
    ax.set_box_aspect((1.15, 1.0, 0.82))
    for tick in ax.get_xticklabels() + ax.get_yticklabels() + ax.get_zticklabels():
        tick.set_fontfamily("DejaVu Serif")
        tick.set_fontsize(9)
    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis.pane.set_facecolor((1, 1, 1, 0))
        axis.pane.set_edgecolor("black")
        axis._axinfo["grid"]["linewidth"] = 0.35
        axis._axinfo["grid"]["color"] = (0, 0, 0, 0.12)


def apply_reference_legend(fig, handles, labels, anchor=(0.98, 0.90)):
    """Place a simple square-swatch legend like the reference image."""
    return fig.legend(
        handles,
        labels,
        loc="upper right",
        bbox_to_anchor=anchor,
        frameon=False,
        prop={"family": "DejaVu Serif", "size": 12},
        handlelength=0.8,
        handletextpad=0.5,
        borderaxespad=0.0,
    )


这里定义“伪 DFT”势能面：水分子的能量由两个 O-H 键长、H-O-H 角度和 H-H 排斥项决定，参考力由能量对坐标的自动微分得到。




In [ ]:
# Toy potential parameters
R0 = 1.0
THETA0 = math.radians(104.5)
D_E = 5.0
A_MORSE = 2.5
K_ANGLE = 1.5

def compute_internal_coords_torch(coords, sort_h=False):
    """Compute r1, r2, theta from coords with shape (..., 3, 3)."""
    o = coords[..., 0, :]
    h1 = coords[..., 1, :]
    h2 = coords[..., 2, :]
    v1 = h1 - o
    v2 = h2 - o
    r1 = torch.linalg.norm(v1, dim=-1).clamp_min(1e-8)
    r2 = torch.linalg.norm(v2, dim=-1).clamp_min(1e-8)
    dot = (v1 * v2).sum(dim=-1)
    cross = torch.linalg.cross(v1, v2, dim=-1)
    cross_norm = torch.linalg.norm(cross, dim=-1)
    theta = torch.atan2(cross_norm, dot)
    if sort_h:
        rr = torch.sort(torch.stack([r1, r2], dim=-1), dim=-1).values
        return rr[..., 0], rr[..., 1], theta
    return r1, r2, theta


def toy_energy_torch(coords):
    """Reference toy potential energy for 3D water-like structures."""
    r1, r2, theta = compute_internal_coords_torch(coords, sort_h=False)
    morse1 = D_E * (1.0 - torch.exp(-A_MORSE * (r1 - R0)))**2
    morse2 = D_E * (1.0 - torch.exp(-A_MORSE * (r2 - R0)))**2
    angle = K_ANGLE * (theta - THETA0)**2
    rhh = torch.linalg.norm(coords[..., 1, :] - coords[..., 2, :], dim=-1).clamp_min(1e-8)
    repulsion = 0.02 / (rhh**6 + 1e-6)
    return morse1 + morse2 + angle + repulsion


def reference_energy_and_forces(coords_np):
    """Compute pseudo-DFT energy and forces with torch autograd."""
    coords = torch.tensor(coords_np, dtype=DTYPE, requires_grad=True)
    energy = toy_energy_torch(coords)
    grad = torch.autograd.grad(energy.sum(), coords)[0]
    forces = -grad
    return energy.detach().numpy(), forces.detach().numpy()


def _sample_angle_ood(n):
    """Sample low-angle and high-angle OOD structures."""
    half = n // 2
    low = np.random.uniform(np.deg2rad(55), np.deg2rad(80), size=half)
    high = np.random.uniform(np.deg2rad(140), np.deg2rad(170), size=n - half)
    theta = np.concatenate([low, high])
    np.random.shuffle(theta)
    return theta


def generate_water_like_dataset(n, mode="ID"):
    """Generate 3D water-like coordinates and metadata for ID or OOD modes."""
    if mode == "ID":
        r1 = np.random.normal(1.0, 0.05, size=n)
        r2 = np.random.normal(1.0, 0.05, size=n)
        theta = np.random.normal(np.deg2rad(104.5), np.deg2rad(5.0), size=n)
    elif mode == "OOD_highT":
        r1 = np.random.normal(1.0, 0.18, size=n)
        r2 = np.random.normal(1.0, 0.18, size=n)
        theta = np.random.normal(np.deg2rad(104.5), np.deg2rad(20.0), size=n)
    elif mode == "OOD_stretch":
        r1 = np.random.uniform(1.3, 1.8, size=n)
        r2 = np.random.normal(1.0, 0.06, size=n)
        theta = np.random.normal(np.deg2rad(104.5), np.deg2rad(6.0), size=n)
    elif mode == "OOD_compress":
        r1 = np.random.uniform(0.65, 0.85, size=n)
        r2 = np.random.normal(1.0, 0.06, size=n)
        theta = np.random.normal(np.deg2rad(104.5), np.deg2rad(6.0), size=n)
    elif mode == "OOD_angle":
        r1 = np.random.normal(1.0, 0.06, size=n)
        r2 = np.random.normal(1.0, 0.06, size=n)
        theta = _sample_angle_ood(n)
    else:
        raise ValueError(f"Unknown mode: {mode}")

    r1 = np.clip(r1, 0.45, 2.2)
    r2 = np.clip(r2, 0.45, 2.2)
    theta = np.clip(theta, np.deg2rad(35), np.deg2rad(175))

    coords = np.zeros((n, 3, 3), dtype=np.float32)
    coords[:, 0, :] = 0.0
    coords[:, 1, 0] = r1 * np.cos(-theta / 2.0)
    coords[:, 1, 1] = r1 * np.sin(-theta / 2.0)
    coords[:, 2, 0] = r2 * np.cos(theta / 2.0)
    coords[:, 2, 1] = r2 * np.sin(theta / 2.0)
    # z coordinates are initially zero: a planar molecule embedded in 3D Cartesian space.
    return coords


def internal_coords_numpy(coords):
    """Compute internal coordinates as numpy arrays for plotting and tables."""
    with torch.no_grad():
        t = torch.tensor(coords, dtype=DTYPE)
        r1, r2, theta = compute_internal_coords_torch(t)
    return r1.numpy(), r2.numpy(), theta.numpy()


这里生成训练集、ID 测试集和四类 OOD 数据。训练集只覆盖接近平衡的水分子构型；四类 OOD 分别测试高温扰动、键拉伸、键压缩和键角偏离时模型是否仍然可靠。




In [ ]:
# Generate train, ID test, and OOD test sets.
train_coords = generate_water_like_dataset(500, "ID")
id_test_coords = generate_water_like_dataset(100, "ID")
ood_modes = ["OOD_highT", "OOD_stretch", "OOD_compress", "OOD_angle"]
ood_coords_by_mode = {m: generate_water_like_dataset(100, m) for m in ood_modes}
ood_coords = np.concatenate([ood_coords_by_mode[m] for m in ood_modes], axis=0)

all_coords = np.concatenate([train_coords, id_test_coords, ood_coords], axis=0)
all_energy, all_forces = reference_energy_and_forces(all_coords)

n_train = len(train_coords)
n_id = len(id_test_coords)
train_energy = all_energy[:n_train]
train_forces = all_forces[:n_train]
id_test_energy = all_energy[n_train:n_train+n_id]
id_test_forces = all_forces[n_train:n_train+n_id]
ood_energy = all_energy[n_train+n_id:]
ood_forces = all_forces[n_train+n_id:]

splits = ["train"] * n_train + ["ID_test"] * n_id + ["OOD_test"] * len(ood_coords)
ood_types = ["ID"] * (n_train + n_id) + sum(([m] * 100 for m in ood_modes), [])
r1_all, r2_all, theta_all = internal_coords_numpy(all_coords)
metadata = pd.DataFrame({
    "structure_id": np.arange(len(all_coords)),
    "split": splits,
    "ood_type": ood_types,
    "r1": r1_all,
    "r2": r2_all,
    "theta_rad": theta_all,
    "theta_deg": np.rad2deg(theta_all),
    "energy": all_energy,
})
metadata.head()


先观察数据覆盖范围。三维构型空间由两个 O-H 键长和 H-O-H 夹角组成；下方展示几个代表性水分子的球棍结构，帮助把构型空间中的点和真实分子形状联系起来。


In [ ]:
plot_meta = metadata.copy()
plot_meta["category"] = np.where(plot_meta["split"].eq("train"), "train", plot_meta["ood_type"])
plot_meta.loc[plot_meta["split"].eq("ID_test"), "category"] = "ID_test"
category_order = ["train", "ID_test", "OOD_highT", "OOD_stretch", "OOD_compress", "OOD_angle"]
color_map = {
    "train": "#1f77b4",
    "ID_test": "#2ca02c",
    "OOD_highT": "#d62728",
    "OOD_stretch": "#9467bd",
    "OOD_compress": "#ff7f0e",
    "OOD_angle": "#e377c2",
}

# py3Dmol normally loads 3Dmol.js from a CDN. Some cloud workspaces block widgets,
# but still allow direct py3Dmol HTML output. A vendored local JS bundle is used when available.
THREEDMOL_JS_PATH = Path("assets/3Dmol-min.js")
if THREEDMOL_JS_PATH.exists():
    THREEDMOL_JS_URI = "data:text/javascript;base64," + base64.b64encode(THREEDMOL_JS_PATH.read_bytes()).decode("ascii")
else:
    THREEDMOL_JS_URI = "https://cdn.jsdelivr.net/npm/3dmol@2.5.4/build/3Dmol-min.js"


def coords_to_xyz_block(coords, title="water toy"):
    """Convert one 3-atom water-like structure to XYZ text for py3Dmol."""
    lines = ["3", title]
    for element, xyz in zip(["O", "H", "H"], coords):
        lines.append(f"{element} {xyz[0]:.6f} {xyz[1]:.6f} {xyz[2]:.6f}")
    return "\n".join(lines)


def show_py3dmol_structure(structure_id, width=360, height=300):
    """Show one selected structure with py3Dmol ball-and-stick rendering."""
    sid = int(structure_id)
    coords = all_coords[sid]
    row = metadata.loc[metadata["structure_id"].eq(sid)].iloc[0]
    title = f"structure_id={sid}, {row['ood_type']}"
    print(f"structure_id={sid} | split={row['split']} | type={row['ood_type']} | r1={row['r1']:.3f} | r2={row['r2']:.3f} | theta={row['theta_deg']:.1f} deg")
    view = py3Dmol.view(width=width, height=height, js=THREEDMOL_JS_URI)
    view.addModel(coords_to_xyz_block(coords, title=title), "xyz")
    view.setStyle({"stick": {"radius": 0.10}, "sphere": {"radius": 0.40}})
    view.zoomTo()
    view.show()


def axis_range_with_padding(values, pad_fraction=0.05):
    """Return an axis range with a small visual padding."""
    lo = float(np.nanmin(values))
    hi = float(np.nanmax(values))
    pad = max((hi - lo) * pad_fraction, 1e-6)
    return [lo - pad, hi + pad]


fig = plt.figure(figsize=(9.4, 6.0))
ax = fig.add_subplot(111, projection="3d")
ax.set_position([0.12, 0.10, 0.60, 0.65])
for category in category_order:
    group = plot_meta[plot_meta["category"] == category]
    size = 16 if category != "train" else 10
    alpha = 0.78 if category != "train" else 0.45
    ax.scatter(group["r1"], group["r2"], group["theta_deg"], s=size, alpha=alpha, c=color_map[category], label=category, depthshade=False)
ax.view_init(elev=24, azim=-52)
ax.set_xlim(axis_range_with_padding(plot_meta["r1"]))
ax.set_ylim(axis_range_with_padding(plot_meta["r2"]))
ax.set_zlim(axis_range_with_padding(plot_meta["theta_deg"]))
apply_reference_3d_style(fig, ax, "Configuration Space Coverage", "r1 = |H1 - O|", "r2 = |H2 - O|", "theta (degree)")
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.855), ncol=3, frameon=False, prop={"family": "DejaVu Serif", "size": 9})
# Manual axes position above keeps the 3D box compact and leaves room for z-label.
# Avoid tight_layout here because it tends to crop 3D labels in notebook frontends.
plt.show()

representative_ids = [
    int(metadata[metadata["split"].eq("ID_test")]["structure_id"].iloc[0]),
    int(metadata[metadata["ood_type"].eq("OOD_highT")]["structure_id"].iloc[0]),
    int(metadata[metadata["ood_type"].eq("OOD_stretch")]["structure_id"].iloc[0]),
    int(metadata[metadata["ood_type"].eq("OOD_compress")]["structure_id"].iloc[0]),
    int(metadata[metadata["ood_type"].eq("OOD_angle")]["structure_id"].iloc[0]),
]
print("下方展示几个代表性构型。可以用鼠标拖动旋转分子、滚轮缩放，检查键长和键角变化。")
for sid in representative_ids:
    show_py3dmol_structure(sid, width=330, height=260)


## Section 2. 模型模块：对比原始坐标与物理不变特征

这一节训练两个 toy MLIP，用来展示表示方式对可靠性的影响。

- **RawCoordNet** 直接使用 9 个笛卡尔坐标作为输入，模型可以拟合训练集，但容易依赖训练时的坐标系。
- **InvariantFeatureNet** 使用 `[sort(r1, r2), theta]` 作为输入，显式加入平移、旋转和同种 H 置换不变性。

训练目标同时包含能量 MSE 和力 MSE，所以模型要同时拟合势能面的值和梯度。这里的重点不是比较网络大小，而是比较“是否把物理对称性放进表示里”。



这里定义两个模型进行对比：RawCoordNet 直接使用笛卡尔坐标，InvariantFeatureNet 使用键长和角度等不变特征。对比重点是物理对称性对模型行为的影响。




In [ ]:
class RawCoordNet(nn.Module):
    """MLP that predicts energy directly from flattened 3D Cartesian coordinates."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(9, 128), nn.SiLU(),
            nn.Linear(128, 128), nn.SiLU(),
            nn.Linear(128, 128), nn.SiLU(),
            nn.Linear(128, 1),
        )

    def forward(self, coords):
        x = coords.reshape(coords.shape[0], -1)
        return self.net(x).squeeze(-1)


class InvariantFeatureNet(nn.Module):
    """MLP that predicts energy from rotation/translation invariant features."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 128), nn.SiLU(),
            nn.Linear(128, 128), nn.SiLU(),
            nn.Linear(128, 128), nn.SiLU(),
            nn.Linear(128, 1),
        )

    def forward(self, coords):
        r1, r2, theta = compute_internal_coords_torch(coords, sort_h=True)
        x = torch.stack([r1, r2, theta], dim=-1)
        return self.net(x).squeeze(-1)


def set_seed(seed):
    """Set random seeds for reproducible training."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def train_energy_model(model, coords_np, energy_np, forces_np, epochs=300, batch_size=64, lr=5e-4, seed=0, force_weight=1.0, sample_weights=None, desc="training"):
    """Train a neural potential with energy and force losses."""
    set_seed(seed)
    model.to(DEVICE)
    coords = torch.tensor(coords_np, dtype=DTYPE)
    energy = torch.tensor(energy_np, dtype=DTYPE)
    forces = torch.tensor(forces_np, dtype=DTYPE)
    if sample_weights is None:
        weights = torch.ones(len(coords), dtype=DTYPE)
    else:
        weights = torch.tensor(sample_weights, dtype=DTYPE)
        weights = weights / weights.mean()
    loader = DataLoader(TensorDataset(coords, energy, forces, weights), batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    epoch_iter = range(epochs)
    if tqdm is not None:
        epoch_iter = tqdm(epoch_iter, desc=desc, leave=False)
    for epoch in epoch_iter:
        total = 0.0
        for xb, yb, fb, wb in loader:
            xb = xb.detach().clone().requires_grad_(True)
            optimizer.zero_grad()
            pred = model(xb)
            grad = torch.autograd.grad(pred.sum(), xb, create_graph=True)[0]
            pred_forces = -grad
            energy_loss = torch.mean(wb * (pred - yb)**2)
            force_loss = torch.mean(wb[:, None, None] * (pred_forces - fb)**2)
            loss = energy_loss + force_weight * force_loss
            loss.backward()
            optimizer.step()
            total += loss.item() * len(xb)
        epoch_loss = total / len(coords)
        history.append(epoch_loss)
        if tqdm is not None:
            epoch_iter.set_postfix(loss=f"{epoch_loss:.3e}")
        elif (epoch + 1) % max(1, epochs // 5) == 0 or epoch == 0:
            print(f"{desc}: epoch {epoch + 1:4d}/{epochs}, loss = {epoch_loss:.4e}")
    return history


def energy_mae(model, coords_np, energy_np):
    """Compute energy MAE."""
    model.eval()
    with torch.no_grad():
        coords = torch.tensor(coords_np, dtype=DTYPE)
        pred = model(coords).cpu().numpy()
    return np.mean(np.abs(pred - energy_np))








这里开始训练两个势函数。loss 里同时包含能量和力，所以模型要同时拟合势能面的值和梯度。




In [ ]:
raw_model = RawCoordNet()
inv_model = InvariantFeatureNet()

raw_history = train_energy_model(raw_model, train_coords, train_energy, train_forces, epochs=500, seed=1, desc="RawCoordNet")
inv_history = train_energy_model(inv_model, train_coords, train_energy, train_forces, epochs=500, seed=2, desc="InvariantFeatureNet")

raw_id_mae = energy_mae(raw_model, id_test_coords, id_test_energy)
inv_id_mae = energy_mae(inv_model, id_test_coords, id_test_energy)

print(f"RawCoordNet ID test energy MAE:       {raw_id_mae:.5f}")
print(f"InvariantFeatureNet ID energy MAE:    {inv_id_mae:.5f}")








这张图用于检查训练是否正常收敛。两个模型的 loss 不需要完全一致，后续会进一步比较它们在对称性和 OOD 测试中的表现。




In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 4.0))
ax.plot(raw_history, label="RawCoordNet", color=MODEL_COLORS["RawCoordNet"], linewidth=2.0)
ax.plot(inv_history, label="InvariantFeatureNet", color=MODEL_COLORS["InvariantFeatureNet"], linewidth=2.0)
ax.set_yscale("log")
apply_reference_axis_style(ax, "Training Loss", "epoch", "Energy + force loss")
ax.legend(frameon=False, loc="upper right", prop={"family": "DejaVu Serif", "size": 10}, handlelength=1.8)
plt.tight_layout()
save_figure(fig, "01_training_loss.png")
plt.show()


## Section 3. 预测模块：从能量自动微分得到力

MLIP 通常先预测势能面，再通过能量梯度得到力：

$$
F_i = -\frac{\partial E}{\partial R_i}
$$

这一节把模型预测的能量转化为力，并在 ID 与各类 OOD 构型上统计 energy RMSE/MAE 和 force RMSE/MAE。核心观察是：ID 上的误差低，并不自动意味着 OOD 区域可靠。




这里先用模型预测能量，再通过 autograd 对坐标求梯度得到预测力。表格和柱状图用于比较 ID 与 OOD 条件下的能量和力误差。




In [ ]:
def predict_energy_and_forces(model, coords_np):
    """Predict energies and forces from a differentiable energy model."""
    model.eval()
    coords = torch.tensor(coords_np, dtype=DTYPE, requires_grad=True)
    energy = model(coords)
    grad = torch.autograd.grad(energy.sum(), coords, create_graph=False)[0]
    forces = -grad
    return energy.detach().numpy(), forces.detach().numpy()


def force_mae(pred_forces, ref_forces):
    """Mean absolute error over all force components."""
    return np.mean(np.abs(pred_forces - ref_forces))


def rmse(pred, ref):
    """Root mean squared error over all entries."""
    return float(np.sqrt(np.mean((pred - ref)**2)))


def evaluate_model_by_group(model, name):
    """Evaluate energy and force MAE for ID test and each OOD group."""
    rows = []
    groups = [("ID_test", id_test_coords, id_test_energy, id_test_forces)]
    start = 0
    for mode in ood_modes:
        end = start + 100
        groups.append((mode, ood_coords[start:end], ood_energy[start:end], ood_forces[start:end]))
        start = end
    for group, coords, e_ref, f_ref in groups:
        e_pred, f_pred = predict_energy_and_forces(model, coords)
        rows.append({
            "model": name,
            "group": group,
            "energy_MAE": np.mean(np.abs(e_pred - e_ref)),
            "energy_RMSE": rmse(e_pred, e_ref),
            "force_MAE": force_mae(f_pred, f_ref),
            "force_RMSE": rmse(f_pred, f_ref),
        })
    return pd.DataFrame(rows)

metrics = pd.concat([
    evaluate_model_by_group(raw_model, "RawCoordNet"),
    evaluate_model_by_group(inv_model, "InvariantFeatureNet"),
], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(13.8, 4.9))
groups = ["ID_test"] + ood_modes
x = np.arange(len(groups))
width = 0.34
bar_groups = {"energy": [], "force": []}
for offset, model_name in [(-width/2, "RawCoordNet"), (width/2, "InvariantFeatureNet")]:
    sub = metrics[metrics["model"] == model_name].set_index("group").loc[groups]
    bar_groups["energy"].append(axes[0].bar(x + offset, sub["energy_RMSE"], width, label=model_name, color=BAR_COLORS[model_name], edgecolor="white", linewidth=0.6))
    bar_groups["force"].append(axes[1].bar(x + offset, sub["force_RMSE"], width, label=model_name, color=BAR_COLORS[model_name], edgecolor="white", linewidth=0.6))
for ax, ylabel, title in [
    (axes[0], "RMSE", "RMSE of Energy"),
    (axes[1], "RMSE", "RMSE of Force"),
]:
    ax.set_yscale("log")
    apply_reference_bar_style(ax, title, ylabel, x, groups, rotation=28)

all_energy_bar_values = []
for bars in bar_groups["energy"]:
    all_energy_bar_values.extend(label_log_bars(axes[0], bars))
expand_log_ylim_for_labels(axes[0], all_energy_bar_values, factor=2.8)
all_force_bar_values = []
for bars in bar_groups["force"]:
    all_force_bar_values.extend(label_log_bars(axes[1], bars))
expand_log_ylim_for_labels(axes[1], all_force_bar_values, factor=2.8)

handles, labels = axes[0].get_legend_handles_labels()
fig.suptitle("ID/OOD Error by Model", y=0.97, fontfamily="DejaVu Serif", fontsize=19)
apply_reference_legend(fig, handles, labels, anchor=(0.985, 0.86))
plt.tight_layout(rect=[0, 0, 0.88, 0.86], w_pad=3.0)
save_figure(fig, "02_id_ood_rmse_by_model.png")
plt.show()

metrics.round(4)

## Section 4. 对称性模块：检查能量不变性与力等变性

物理上，总能量应当对整体平移和旋转不变。力不是旋转不变的向量，而是应当随构型一起旋转，也就是旋转等变：

$$
E(Rx) = E(x), \quad F(Rx) = R F(x)
$$

这一节对 RawCoordNet 和 InvariantFeatureNet 分别做平移、旋转和力等变性检查。这个 sanity check 可以在真正跑大规模模拟前，提前发现表示方式中的明显物理问题。



这里检查物理对称性。能量应满足平移和旋转不变性；力不是旋转不变的，而应随分子构型旋转，即满足旋转等变性。




In [ ]:
def rotation_matrix_3d(axis, angle_rad):
    """Build a 3D rotation matrix from an axis and angle."""
    axis = np.asarray(axis, dtype=np.float32)
    axis = axis / np.linalg.norm(axis)
    x, y, z = axis
    c = np.cos(angle_rad)
    s = np.sin(angle_rad)
    C = 1.0 - c
    return np.array([
        [c + x*x*C,     x*y*C - z*s, x*z*C + y*s],
        [y*x*C + z*s,   c + y*y*C,   y*z*C - x*s],
        [z*x*C - y*s,   z*y*C + x*s, c + z*z*C],
    ], dtype=np.float32)


def rotate_coords(coords_np, angle_rad):
    """Rotate 3D coordinates around an arbitrary 3D axis by angle_rad."""
    r = rotation_matrix_3d(axis=[1.0, 1.0, 0.5], angle_rad=angle_rad)
    return coords_np @ r.T


def symmetry_checks(model, coords_single):
    """Compute translation invariance, rotation invariance, and force equivariance errors."""
    x = coords_single[None, :, :]
    translation = np.array([0.7, -1.2, 0.4], dtype=np.float32)
    xt = x + translation
    angle = np.deg2rad(60.0)
    xr = rotate_coords(x, angle)

    e, f = predict_energy_and_forces(model, x)
    et, _ = predict_energy_and_forces(model, xt)
    er, fr = predict_energy_and_forces(model, xr)
    rf = rotate_coords(f, angle)

    return {
        "translation_energy_error": float(np.abs(et[0] - e[0])),
        "rotation_energy_error": float(np.abs(er[0] - e[0])),
        "force_equivariance_error": float(np.linalg.norm(fr[0] - rf[0], axis=1).mean()),
    }

x0 = id_test_coords[0]
symmetry_table = pd.DataFrame([
    {"model": "RawCoordNet", **symmetry_checks(raw_model, x0)},
    {"model": "InvariantFeatureNet", **symmetry_checks(inv_model, x0)},
])

fig, ax = plt.subplots(figsize=(8.8, 4.8))
symmetry_metrics = ["translation_energy_error", "rotation_energy_error", "force_equivariance_error"]
metric_labels = ["translation E", "rotation E", "force equiv."]
x = np.arange(len(symmetry_metrics))
width = 0.34
bar_groups = []
for offset, model_name in [(-width/2, "RawCoordNet"), (width/2, "InvariantFeatureNet")]:
    row = symmetry_table[symmetry_table["model"] == model_name].iloc[0]
    values = np.maximum([row[m] for m in symmetry_metrics], 1e-12)
    bar_groups.append(ax.bar(x + offset, values, width, label=model_name, color=BAR_COLORS[model_name], edgecolor="white", linewidth=0.6))
ax.set_yscale("log")
all_bar_values = []
for bars in bar_groups:
    all_bar_values.extend(label_log_bars(ax, bars, fmt="{:.1e}"))
expand_log_ylim_for_labels(ax, all_bar_values, factor=3.0)
apply_reference_bar_style(ax, "Symmetry Error Check", "Error", x, metric_labels)
handles, labels = ax.get_legend_handles_labels()
apply_reference_legend(fig, handles, labels, anchor=(0.98, 0.78))
if ax.legend_ is not None:
    ax.legend_.remove()
plt.tight_layout(rect=[0, 0, 0.73, 0.95])
save_figure(fig, "03_symmetry_error_check.png")
plt.show()

symmetry_table.round(6)


## Section 5. 诊断模块：识别 OOD 失效区域

这一节聚焦对称性更合理的 InvariantFeatureNet，比较它在 ID 和四类 OOD 构型上的误差，并找出 force RMSE 最大的结构。

我们把高误差构型重新放回构型空间中观察：如果它们集中在训练数据覆盖不足的区域，就说明问题主要来自数据盲区，而不只是模型没有训练好。



这里聚焦 InvariantFeatureNet，将它在 ID 和各类 OOD 构型上的误差分组统计。高误差结构有助于定位模型在哪些构型区域失效。




In [ ]:
def per_structure_force_mae(pred_forces, ref_forces):
    """Compute per-structure force MAE over atoms and coordinates."""
    return np.mean(np.abs(pred_forces - ref_forces), axis=(1, 2))


def per_structure_force_rmse(pred_forces, ref_forces):
    """Compute per-structure force RMSE over atoms and coordinates."""
    return np.sqrt(np.mean((pred_forces - ref_forces)**2, axis=(1, 2)))

inv_eval_metrics = metrics[metrics["model"] == "InvariantFeatureNet"].copy()
inv_eval_metrics


这里先计算每个 OOD 构型的 force MAE 和 force RMSE，并找出误差最大的结构。下一张图会把这些高误差点放回构型空间中观察。




In [ ]:
e_ood_pred, f_ood_pred = predict_energy_and_forces(inv_model, ood_coords)
per_ood_force_mae = per_structure_force_mae(f_ood_pred, ood_forces)
per_ood_force_rmse = per_structure_force_rmse(f_ood_pred, ood_forces)
ood_meta = metadata[metadata["split"] == "OOD_test"].copy().reset_index(drop=True)
ood_meta["force_mae"] = per_ood_force_mae
ood_meta["force_rmse"] = per_ood_force_rmse

top10 = (ood_meta.sort_values("force_rmse", ascending=False)
         [["structure_id", "ood_type", "r1", "r2", "theta_deg", "force_mae", "force_rmse"]]
         .head(10))

print("已计算每个 OOD 构型的 force MAE / RMSE，用于定位高误差区域。")


这里将高误差 OOD 构型映射回构型空间，并展示 force RMSE 最高的几个分子结构。这样可以直观看到模型最不可靠的构型是否位于训练数据覆盖不足的区域。


In [ ]:
train_meta = metadata[metadata["split"] == "train"]
id_meta = metadata[metadata["split"] == "ID_test"]
high_error_ids = set(top10["structure_id"])
high_error = ood_meta[ood_meta["structure_id"].isin(high_error_ids)].copy()
other_ood = ood_meta[~ood_meta["structure_id"].isin(high_error_ids)].copy()

fig = plt.figure(figsize=(9.4, 6.0))
ax3d = fig.add_subplot(111, projection="3d")
ax3d.set_position([0.12, 0.10, 0.60, 0.65])
ax3d.scatter(train_meta["r1"], train_meta["r2"], train_meta["theta_deg"], s=10, alpha=0.25, c="#1f77b4", label="train ID", depthshade=False)
ax3d.scatter(id_meta["r1"], id_meta["r2"], id_meta["theta_deg"], s=10, alpha=0.25, c="#2ca02c", label="ID test", depthshade=False)
ax3d.scatter(other_ood["r1"], other_ood["r2"], other_ood["theta_deg"], s=10, alpha=0.18, c="#9e9e9e", label="other OOD", depthshade=False)
ax3d.scatter(high_error["r1"], high_error["r2"], high_error["theta_deg"], s=58, alpha=0.95, c="#d62728", edgecolors="white", linewidths=0.8, label="top error OOD", depthshade=False)
ax3d.view_init(elev=24, azim=-52)
all_for_axes = pd.concat([train_meta, id_meta, ood_meta], ignore_index=True)
ax3d.set_xlim(axis_range_with_padding(all_for_axes["r1"]))
ax3d.set_ylim(axis_range_with_padding(all_for_axes["r2"]))
ax3d.set_zlim(axis_range_with_padding(all_for_axes["theta_deg"]))
apply_reference_3d_style(fig, ax3d, "High-Error OOD Points", "r1", "r2", "theta (degree)")
handles, labels = ax3d.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.855), ncol=4, frameon=False, prop={"family": "DejaVu Serif", "size": 9})
# Manual axes position above keeps the 3D box compact and leaves room for z-label.
# Avoid tight_layout here because it tends to crop 3D labels in notebook frontends.
save_figure(fig, "04_high_error_ood_locations.png")
plt.show()

print("下方展示 force RMSE 最高的几个 OOD 构型。可以拖动和缩放分子，观察它们对应的键长和键角。")
for sid in top10["structure_id"].head(5):
    show_py3dmol_structure(int(sid), width=330, height=260)


## Section 6. 主动学习模块：用 committee disagreement 选点

主动学习的思想是：如果 DFT 标注很贵，不应随机补数据，而应优先选择模型分歧较高、可能更有信息量的构型。

这里训练 4 个不同随机种子的 InvariantFeatureNet，假装 OOD pool 没有标签。对每个 pool 构型，我们计算 4 个模型预测力的分歧：

$$
uncertainty = mean(std(F_{committee}))
$$

为了模拟主动学习中的无标签选点，本节只用 committee disagreement 做选点：每个 OOD 区域各选不确定性最高的 5 个构型，总共 20 个点。toy 数据里的伪 DFT 标签只在选点后用于模拟新增标注和评估改进效果。



这里模拟主动学习中的 committee disagreement。多个模型对同一构型的力预测分歧越大，说明该构型的不确定性越高，越值得优先作为补充标注候选。下面会按每类 OOD 的不确定性排序选点，而不是使用真实 force RMSE。




In [ ]:
committee = [inv_model]
committee_histories = [inv_history]
for seed in [10, 11, 12]:
    set_seed(seed)
    m = InvariantFeatureNet()
    hist = train_energy_model(m, train_coords, train_energy, train_forces, epochs=250, seed=seed, desc=f"committee seed {seed}")
    committee.append(m)
    committee_histories.append(hist)

force_predictions = []
energy_predictions = []
for m in committee:
    ep, fp = predict_energy_and_forces(m, ood_coords)
    energy_predictions.append(ep)
    force_predictions.append(fp)
force_predictions = np.stack(force_predictions, axis=0)  # (n_models, n_structures, 3, 3)
uncertainty = force_predictions.std(axis=0).mean(axis=(1, 2))

pool_table = ood_meta.copy()
pool_table["uncertainty"] = uncertainty

N_COMMITTEE_PER_OOD_TYPE = 5
selected = (pool_table.sort_values("uncertainty", ascending=False)
            .groupby("ood_type", group_keys=False)
            .head(N_COMMITTEE_PER_OOD_TYPE)
            .sort_values(["ood_type", "uncertainty"], ascending=[True, False]))
print(f"本轮按 committee disagreement 选出 {len(selected)} 个待补充标注的 OOD 构型。")


## Section 7. 闭环模块：进行一次主动学习数据回流

这一节把上一步由 committee disagreement 选出的 OOD 构型加入训练集，并从头训练一个新的 InvariantFeatureNet。

主动学习的价值不是简单增加数据量，而是把标注预算优先用在模型分歧较高、可能更有信息量的区域。最后的对比图展示数据回流前后 OOD force RMSE 的变化；少量 OOD 点回流也可能牺牲部分 ID 区域精度，这是主动学习中需要监控的 trade-off。



最后进行一次数据回流：从每类 OOD 中选择 committee disagreement 最高的构型，加入训练集后从头训练，并比较 ID 与 OOD force RMSE 的变化。




In [ ]:
selected_idx = selected.index.to_numpy()
augmented_coords = np.concatenate([train_coords, ood_coords[selected_idx]], axis=0)
augmented_energy = np.concatenate([train_energy, ood_energy[selected_idx]], axis=0)
augmented_forces = np.concatenate([train_forces, ood_forces[selected_idx]], axis=0)
sample_weights = np.ones(len(augmented_coords), dtype=np.float32)
sample_weights[-len(selected_idx):] = 1.0  # conservative: avoid overfitting to only 20 new labels

al_model = InvariantFeatureNet()
al_history = train_energy_model(al_model, augmented_coords, augmented_energy, augmented_forces, epochs=500, seed=123, sample_weights=sample_weights, desc="active-learning retrain")
al_metrics = evaluate_model_by_group(al_model, "InvariantFeatureNet + 20 committee AL points")

before_after = pd.concat([
    inv_eval_metrics.assign(model="before"),
    al_metrics.assign(model="after"),
], ignore_index=True)

fig, ax = plt.subplots(figsize=(8.4, 4.9))
x = np.arange(len(inv_eval_metrics["group"]))
width = 0.34
before = before_after[before_after["model"] == "before"]
after = before_after[before_after["model"] == "after"]
bars_before = ax.bar(x - width/2, before["force_RMSE"], width, label="before", color=BAR_COLORS["before"], edgecolor="white", linewidth=0.6)
bars_after = ax.bar(x + width/2, after["force_RMSE"], width, label="after + 20 labels", color=BAR_COLORS["after"], edgecolor="white", linewidth=0.6)
ax.set_yscale("log")
all_bar_values = []
all_bar_values.extend(label_log_bars(ax, bars_before))
all_bar_values.extend(label_log_bars(ax, bars_after))
expand_log_ylim_for_labels(ax, all_bar_values, factor=2.8)
apply_reference_bar_style(ax, "Active-Learning Update", "RMSE of Force", x, before["group"], rotation=28)

handles, labels = ax.get_legend_handles_labels()
apply_reference_legend(fig, handles, labels, anchor=(0.98, 0.82))
if ax.legend_ is not None:
    ax.legend_.remove()
plt.tight_layout(rect=[0, 0, 0.76, 0.95])
save_figure(fig, "05_active_learning_update.png")
plt.show()

display(al_metrics.round(4))
before_mean = inv_eval_metrics[inv_eval_metrics["group"] != "ID_test"]["force_RMSE"].mean()
after_mean = al_metrics[al_metrics["group"] != "ID_test"]["force_RMSE"].mean()
print(f"Mean OOD force RMSE: before = {before_mean:.4f}, after = {after_mean:.4f}")
print("主动学习不是盲目加数据，而是优先补充模型分歧较高、可能更有信息量的构型；同时也要监控 ID/OOD 之间的精度 trade-off。")


## 总结：我们学到了什么

1. **MLIP 学习的是构型到能量的函数。** 模型先预测能量，再通过能量梯度得到力，因此力误差反映了势能面局部斜率是否可靠。
2. **物理对称性是必要的 sanity check。** 能量应满足平移、旋转和同种原子置换不变性；力应满足旋转等变性。
3. **ID 测试误差低不代表 OOD 可靠。** 当构型进入训练数据覆盖不足的区域，模型可能给出低可信度的力和能量。
4. **主动学习关注数据盲区。** Committee disagreement 可以作为低成本的不确定性指标，帮助优先筛选值得补充 DFT 标注的构型，形成持续验证与数据回流闭环。

可靠的机器学习势函数，不只依赖模型架构，还依赖训练数据对目标构型空间的覆盖，以及持续的验证与数据回流。

